## 환경설정

In [ ]:
# 인증

from google.colab import auth
auth.authenticate_user()
print("인증 완료")

In [ ]:
# 매직 로드

%load_ext google.cloud.bigquery

## stg_withdraw
- 유저가 떠나는 이유를 보여주는 탈퇴 사유 테이블

### 한계점 (명시)
- user_id 없음 + id가 raw_user와 0% 매칭 → 유저 단위 연결 불가
- "누가 탈퇴했나" 분석 불가. 유저 단위 이탈은 ban_status='W' : 추후 EDA 시 재확인 필요
- 탈퇴 기록 70,764건 vs ban W 7,855명 (9배 차이) → 재가입 반복/탈퇴하면서 삭제 추정

## 타입 확인
- `user_id` 가 없음

In [ ]:
%%bigquery df_wd_types --project your-gcp-project
-- 실제 테이블에 어떤 컬럼이 있는지 (명세서 누락 대비)
SELECT column_name, data_type
FROM `your-gcp-project.raw.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'raw_userwithdraw'
ORDER BY ordinal_position;

In [ ]:
df_wd_types

## 유저 추가 확인
- withdraw.id가 `id`인지 확인 결과, 매칭률 0% = 누가 탈퇴한건지 확인 불가
> 유저 데이터와 조인 불가, 집계/사유/추이 분석용 성격의 테이블

In [ ]:
%%bigquery df_wd_idcheck --project your-gcp-project
-- withdraw.id가 유저 id인지 판정 (raw_user와 매칭률)
WITH wd AS (
  SELECT DISTINCT id FROM `your-gcp-project.raw.raw_userwithdraw`
),
matched AS (
  SELECT w.id
  FROM wd w
  JOIN `your-gcp-project.raw.raw_user` u ON u.id = w.id
)
SELECT
  (SELECT COUNT(*) FROM wd)                                        AS withdraw_rows,
  (SELECT COUNT(*) FROM matched)                                  AS matched_in_users,
  ROUND((SELECT COUNT(*) FROM matched) / (SELECT COUNT(*) FROM wd) * 100, 1) AS match_pct
FROM wd
LIMIT 1;

In [ ]:
df_wd_idcheck

## 탈퇴 사유 확인
- "함께 할 친구가 없어서" 20.4% → 축 B(학교 네트워크 임계)
- "40명 못 넘으면 친구가 없어서 떠난다는 가설을, 탈퇴 사유가 증명해줌
- 재밌는 질문이 없어서" 18.6% → 축 D(콘텐츠 피로도)
- 질문 콘텐츠가 이탈 사유라는 직접 증거
- 기타 57%는 추후 확인하기

In [ ]:
%%bigquery df_wd_reason --project your-gcp-project
-- 탈퇴 사유 분포 (이 테이블의 핵심 가치)
SELECT
  reason,
  COUNT(*) AS cnt,
  ROUND(COUNT(*) / SUM(COUNT(*)) OVER () * 100, 1) AS pct
FROM `your-gcp-project.raw.raw_userwithdraw`
GROUP BY reason
ORDER BY cnt DESC
LIMIT 30;

In [ ]:
df_wd_reason

## Table 생성

In [ ]:
%%bigquery --project your-gcp-project
CREATE OR REPLACE VIEW `your-gcp-project.stage.stg_withdraw` AS
WITH src AS (
  SELECT id, reason, created_at
  FROM `your-gcp-project.raw.raw_userwithdraw`
)
SELECT
  id                                 AS withdraw_id,       -- 레코드 PK (유저 id 아님 주의!)
  reason,                                                  -- 원본 사유
  -- 더미(운영/테스트) 제외 플래그
  (reason NOT IN ('admin', 'test'))  AS is_valid_withdraw,
  -- 사유를 분석 축으로 매핑 (서비스 개선 제안용)
  CASE reason
    WHEN '함께 할 친구가 없어서' THEN 'network'
    WHEN '재밌는 질문이 없어서'  THEN 'content'
    WHEN '버그가 너무 많아서'    THEN 'tech'
    WHEN '구독료가 너무 비싸서'  THEN 'price'
    WHEN '기타 이유'            THEN 'etc'
    WHEN '기타'                THEN 'etc'
    ELSE 'dummy'
  END                                AS reason_category,
  DATETIME(created_at, 'Asia/Seoul') AS withdrawn_at       -- 탈퇴 시각 (KST 변환)
FROM src;

### 검증

In [ ]:
%%bigquery --project your-gcp-project
SELECT
  reason_category,
  COUNT(*) AS cnt,
  COUNTIF(NOT is_valid_withdraw) AS dummy_cnt
FROM `your-gcp-project.stage.stg_withdraw`
GROUP BY reason_category
ORDER BY cnt DESC;